미세조정 전후 비교

In [13]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
test_reviews = [
    "This movie is absolutely amazing! Best film I've ever seen.",
    "Terrible waste of time. Worst movie ever made.",
    "It was okay, nothing special but not bad either.",
    "Brilliant acting and stunning visuals throughout!",
    "Boring and predictable plot. Very disappointed."
]

labels = ["긍정", "부정"]

# 미세 조정 전 모델(랜덤초기화)
BERT_MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model_before = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME) 
model_before.eval() # 평가모드
with torch.no_grad():
    for i, review in enumerate(test_reviews):
        inputs = tokenizer(review, return_tensors="pt")
        outputs = model_before(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_label = torch.argmax(probs).item()

        print(f'리뷰 : {i + 1} : {review}')
        print(f'예측 : {labels[pred_label]} (긍정 확률: {probs[pred_label]:.4f})')
        print(f'긍정 : {probs[1]:.4f}, 부정 : {probs[0]:.4f}')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


리뷰 : 1 : This movie is absolutely amazing! Best film I've ever seen.
예측 : 부정 (긍정 확률: 0.5384)
긍정 : 0.5384, 부정 : 0.4616
리뷰 : 2 : Terrible waste of time. Worst movie ever made.
예측 : 부정 (긍정 확률: 0.5794)
긍정 : 0.5794, 부정 : 0.4206
리뷰 : 3 : It was okay, nothing special but not bad either.
예측 : 긍정 (긍정 확률: 0.5146)
긍정 : 0.4854, 부정 : 0.5146
리뷰 : 4 : Brilliant acting and stunning visuals throughout!
예측 : 부정 (긍정 확률: 0.5778)
긍정 : 0.5778, 부정 : 0.4222
리뷰 : 5 : Boring and predictable plot. Very disappointed.
예측 : 부정 (긍정 확률: 0.5613)
긍정 : 0.5613, 부정 : 0.4387


In [15]:
# 학습 - 미세조정
import torch
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForSequenceClassification
train_texts = [
    ("This is wonderful and fantastic!", 1),  # 긍정
    ("Absolutely terrible and awful!", 0),     # 부정
    ("I love this so much!", 1),
    ("Hated it completely!", 0)
]
BERT_MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model_after = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME,num_labels=2)
optimizer = AdamW(model_after.parameters(), lr=2e-5)
model_after.train()
for epoch in range(7):
    total_loss = 0
    for text,label in train_texts:
        optimizer.zero_grad()
        inputs = tokenizer(text,return_tensors='pt')
        labels = torch.tensor([label])
        outputs = model_after(**inputs,labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(train_texts)
    print(f'epoch : {epoch+1}  loss : {avg_loss:.4f}')   

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch : 1  loss : 0.6679
epoch : 2  loss : 0.6230
epoch : 3  loss : 0.6129
epoch : 4  loss : 0.5119
epoch : 5  loss : 0.4544
epoch : 6  loss : 0.3940
epoch : 7  loss : 0.2776


In [ ]:
print(f'미세조정 후')
test_reviews = [
    "This movie is absolutely amazing! Best film I've ever seen.",
    "Terrible waste of time. Worst movie ever made.",
    "It was okay, nothing special but not bad either.",
    "Brilliant acting and stunning visuals throughout!",
    "Boring and predictable plot. Very disappointed."
]

labels = ["긍정", "부정"]

with torch.no_grad():
    for i, review in enumerate(test_reviews):
        inputs = tokenizer(review,return_tensors='pt')
        outputs =  model_after(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits,dim=-1)[0]
        pred_label = torch.argmax(probs).item()        
        print(f'리뷰 : {i+1} : {review}')
        print(f'예측 : {labels[pred_label]} {probs[pred_label]}')
        print(f'긍정 : {probs[0]} | 부정 : {probs[1]}\n')

미세조정 후
리뷰 : 1 : This movie is absolutely amazing! Best film I've ever seen.
예측 : 긍정 0.5163581967353821
긍정 : 0.5163581967353821 | 부정 : 0.48364174365997314

리뷰 : 2 : Terrible waste of time. Worst movie ever made.
예측 : 긍정 0.5673832297325134
긍정 : 0.5673832297325134 | 부정 : 0.4326167404651642

리뷰 : 3 : It was okay, nothing special but not bad either.
예측 : 부정 0.5740005970001221
긍정 : 0.42599937319755554 | 부정 : 0.5740005970001221

리뷰 : 4 : Brilliant acting and stunning visuals throughout!
예측 : 부정 0.5558239817619324
긍정 : 0.44417604804039 | 부정 : 0.5558239817619324

리뷰 : 5 : Boring and predictable plot. Very disappointed.
예측 : 부정 0.5438677072525024
긍정 : 0.45613229274749756 | 부정 : 0.5438677072525024

